# Cleaning the LEIE List

This takes the raw data file from the OIG website on the most recent exclusions list. 

Data updated 02/06/2026 from the 12-2025 public file.

The original data can have multiple rows per entity based on differences in name descriptions, different locations, different exclusion dates, and different exclusion types. 

Data Cleaning Steps:
- Change all data to columns to snake case
- Change the data types of specified columns
- Drop blank columns
- Keep only the most recent name descriptions for each npi
- filter dataset on exclusion date greater than 2020-12-31
- filter dataset on specified exclusion types (the crosswalk and definition of each exclusion type is in our teams Capstone folder
- filter dataset on most recent exclusion only to get 1 per row

Additional Columns
- Add column for number of exclusion dates
- Add column for number of unique exclusions
- Add column for list of unique exclusions
- Add column for number of unique addresses
- fraud_flag which is 1 for all rows

Return a dataset that has null NPIs to possibly backfill later. ("/dsa/groups/casestudycf25/team02/bronze/leie_with_null_npi_clean.csv", index=False)

Return clean dataset of only valid NPIs and exclusion dates greater than 2020
/dsa/groups/casestudycf25/team02/leie_with_valid_npi_clean.csv

In [1]:
# Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Change all headers to snake case
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

In [3]:
# Read in CSV from file path
df = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/LEIE_OIG_Exclusion_List.csv")

df = df.rename(columns={col: to_snake_case(col) for col in df.columns})

df.head(5)

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,lastname,firstname,midname,busname,general,specialty,upin,npi,dob,address,city,state,zip,excltype,excldate,reindate,waiverdate,wvrstate
0,NaN,NaN,NaN,"#1 MARKETING SERVICE, INC",OTHER BUSINESS,SOBER HOME,NaN,0,NaN,239 BRIGHTON BEACH AVENUE,BROOKLYN,NY,11235,1128a1,20200319,0,0,NaN
1,NaN,NaN,NaN,"1 BEST CARE, INC",OTHER BUSINESS,HOME HEALTH AGENCY,NaN,0,NaN,"2161 UNIVERSITY AVENUE W, STE",SAINT PAUL,MN,55114,1128b5,20230518,0,0,NaN
2,NaN,NaN,NaN,101 FIRST CARE PHARMACY INC,OTHER BUSINESS,PHARMACY,NaN,1972902351,NaN,"C/O 609 W 191ST STREET, APT D",NEW YORK,NY,10040,1128b8,20220320,0,0,NaN
3,NaN,NaN,NaN,14 LAWRENCE AVE PHARMACY,PHARMACY,NaN,NaN,0,NaN,14 LAWRENCE AVENUE,SMITHTOWN,NY,11787,1128a1,19880830,0,0,NaN
4,NaN,NaN,NaN,143 MEDICAL EQUIPMENT CO,DME COMPANY,DME - OXYGEN,NaN,0,NaN,701 NW 36 AVENUE,MIAMI,FL,33125,1128b7,19970620,0,0,NaN


In [4]:
print(f"Original Dataset Shape: {df.shape}")

Original Dataset Shape: (82709, 18)


In [5]:
# Drop blank columns
df.drop(columns = ['reindate', 'waiverdate', 'wvrstate'], inplace = True) # drop blank columns

In [6]:
# Convert npi
df['npi'] = df['npi'].replace(0, np.nan)

In [7]:
df.dtypes

lastname      object
firstname     object
midname       object
busname       object
general       object
specialty     object
upin          object
npi          float64
dob          float64
address       object
city          object
state         object
zip            int64
excltype      object
excldate       int64
dtype: object

In [8]:
df.head(5)

,lastname,firstname,midname,busname,general,specialty,upin,npi,dob,address,city,state,zip,excltype,excldate
0,NaN,NaN,NaN,"#1 MARKETING SERVICE, INC",OTHER BUSINESS,SOBER HOME,NaN,NaN,NaN,239 BRIGHTON BEACH AVENUE,BROOKLYN,NY,11235,1128a1,20200319
1,NaN,NaN,NaN,"1 BEST CARE, INC",OTHER BUSINESS,HOME HEALTH AGENCY,NaN,NaN,NaN,"2161 UNIVERSITY AVENUE W, STE",SAINT PAUL,MN,55114,1128b5,20230518
2,NaN,NaN,NaN,101 FIRST CARE PHARMACY INC,OTHER BUSINESS,PHARMACY,NaN,1.972902e+09,NaN,"C/O 609 W 191ST STREET, APT D",NEW YORK,NY,10040,1128b8,20220320
3,NaN,NaN,NaN,14 LAWRENCE AVE PHARMACY,PHARMACY,NaN,NaN,NaN,NaN,14 LAWRENCE AVENUE,SMITHTOWN,NY,11787,1128a1,19880830
4,NaN,NaN,NaN,143 MEDICAL EQUIPMENT CO,DME COMPANY,DME - OXYGEN,NaN,NaN,NaN,701 NW 36 AVENUE,MIAMI,FL,33125,1128b7,19970620


## Add Additional Feature Columns

In [9]:
# Add column for number of exclusion dates
df["num_exclusions_alltime"] = df.groupby("npi")["excldate"].transform("nunique")

In [10]:
# Add column for number of unique exclusions
df["num_exclusion_types_alltime"] = df.groupby("npi")["excltype"].transform("nunique")

In [11]:
# Add column for list of unique exclusions
type_map = df.groupby("npi")["excltype"].apply(lambda x: list(x.dropna().unique()))
df["list_exclusion_types_alltime"] = df["npi"].map(type_map)

In [12]:
# Add column for number of unique addresses
df["num_addresses_alltime"] = df.groupby("npi")["address"].transform("nunique")

In [13]:
# Convert to datetime
df["excldate"] = pd.to_datetime(df["excldate"], format="%Y%m%d", errors="coerce")
df["dob"] = pd.to_datetime(df["dob"], format="%Y%m%d", errors="coerce")

In [14]:
# Convert exclusion types to all lower case
df["excltype"] =  df["excltype"].str.lower()

In [15]:
df[df["num_exclusion_types_alltime"] > 2]

,lastname,firstname,midname,busname,general,specialty,upin,npi,dob,address,city,state,zip,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime
16112,CHRISTENSEN,JOHN,PETER,NaN,"PHYSICIAN (MD, DO)",GENERAL PRACTICE,K9897A,1.801839e+09,1951-09-25,2900 N FLAGER DRIVE,WEST PALM BEACH,FL,33407,1128b4,2013-11-20,3.0,3.0,"[1128b4, 1128a2, 1128a1]",3.0
16113,CHRISTENSEN,JOHN,PETER,NaN,"PHYSICIAN (MD, DO)",GENERAL PRACTICE,NaN,1.801839e+09,1951-09-25,"568 NE 255TH STREET, #D63217",CROSS CITY,FL,32628,1128a2,2019-02-20,3.0,3.0,"[1128b4, 1128a2, 1128a1]",3.0
16114,CHRISTENSEN,JOHN,NaN,NaN,"PHYSICIAN (MD, DO)",GENERAL PRACTICE,I53515,1.801839e+09,1951-09-25,P O BOX 019120,MIAMI,FL,33101,1128a1,2017-05-18,3.0,3.0,"[1128b4, 1128a2, 1128a1]",3.0
29497,GOINS,RONDY,DWAYNE,NaN,IND- LIC HC SERV PRO,PODIATRY,U19347,1.508905e+09,1957-12-12,"13201 VICTORIA PARK DR, N",DETROIT,MI,48215,1128b4,2013-02-20,3.0,3.0,"[1128b4, 1128a1, 1128b14]",3.0
29498,GOINS,RONDY,NaN,NaN,IND- LIC HC SERV PRO,PODIATRY,NaN,1.508905e+09,1957-12-12,"P O BOX 1000, #56760-039",MORGANTOWN,WV,26507,1128a1,2023-05-18,3.0,3.0,"[1128b4, 1128a1, 1128b14]",3.0
29499,GOINS,RONDY,DWAYNE,NaN,IND- LIC HC SERV PRO,PODIATRY,NaN,1.508905e+09,1957-12-12,13201 VICTORIA PARK DRIVE N,DETROIT,MI,48215,1128b14,2018-12-20,3.0,3.0,"[1128b4, 1128a1, 1128b14]",3.0
33468,HASPEL,ARTHUR,CARL,NaN,PODIATRY PRACTICE,PODIATRY,T55450,1.225072e+09,1945-05-18,22949 OLD INLET BRIDGE DRIVE,BOCA RATON,FL,33433,1128a4,2003-07-20,3.0,3.0,"[1128a4, 1128b7, 1128a1]",3.0
33469,HASPEL,ARTHUR,CARL,NaN,PODIATRY PRACTICE,PODIATRY,T55450,1.225072e+09,1945-05-18,22949 OLD INLET BRIDGE ROAD,BOCA RATON,FL,33433,1128b7,2010-09-01,3.0,3.0,"[1128a4, 1128b7, 1128a1]",3.0
33470,HASPEL,ARTHUR,CARL,NaN,IND- LIC HC SERV PRO,PODIATRY,T55450,1.225072e+09,1945-05-18,"P O BOX 779800, #30086-074",MIAMI,FL,33177,1128a1,2012-05-20,3.0,3.0,"[1128a4, 1128b7, 1128a1]",3.0


In [16]:
# NPI Dimension

# Filter out null NAME values
npi_dim = df[df["npi"].notna()].copy()

# Sort by ID and DATE descending
npi_dim = npi_dim.sort_values(["npi", "excldate"], ascending=[True, False])

# Keep only the most recent name descriptions for each npi
npi_dim = npi_dim.groupby("npi").first().reset_index()

npi_dim = npi_dim[["npi", "lastname", "firstname", "midname", "busname"]]

npi_dim.head()

,npi,lastname,firstname,midname,busname
0,1.003005e+09,SMITH,ADAM,BRYANT,NaN
1,1.003012e+09,RICE,CHRISTAL,NaN,NaN
2,1.003017e+09,MARTINEZ,BENJAMIN,SETH,NaN
3,1.003028e+09,MESSERLY,RICHARD,CHARLES,NaN
4,1.003035e+09,WARD,SHARON,ROMAINE,NaN


In [17]:
npi_dim.dtypes

npi          float64
lastname      object
firstname     object
midname       object
busname       object
dtype: object

In [18]:
df = df.merge(
    npi_dim[["npi", "lastname", "firstname", "midname", "busname"]],
    on="npi",
    how="left",
    suffixes=("", "_npi")
)

In [19]:
# Change the dataframe to use the dimension descriptions so that the text for the name columns are consistent across the data
df["lastname"]  = df["lastname_npi"].combine_first(df["lastname"])
df["firstname"] = df["firstname_npi"].combine_first(df["firstname"])
df["midname"]   = df["midname_npi"].combine_first(df["midname"])
df["busname"]   = df["busname_npi"].combine_first(df["busname"])

In [20]:
# Drop interim name descr columns
df = df.drop(columns=["lastname_npi", "firstname_npi", "midname_npi", "busname_npi"])

## Filter the Dataset

In [21]:
###################################
# filter dataset on exclusion date greater than 2020
###################################
df_filtered = df[df["excldate"] >= "2020-12-31"].copy()

In [22]:
print(f"New Dataset Shape After Filtering for Exclusion Dates: {df_filtered.shape}")

New Dataset Shape After Filtering for Exclusion Dates: (12349, 19)


In [23]:
###################################
# filter dataset on specified exclusion types
###################################

specified_types = [
    "1128a1", "1128a2", "1128a3",
    "1128b1a", "1128b5", "1128b6", "1128b7",
    "1128c3gi", "1128c3gii",
    "1156",
    "brch cia"
]

df_filtered = df_filtered[df_filtered["excltype"].isin([x.lower() for x in specified_types])]

In [24]:
print(f"New Dataset Shape After Filtering for Exclusion Types: {df_filtered.shape}")

New Dataset Shape After Filtering for Exclusion Types: (7717, 19)


In [25]:
# Create Unique ID for each row

id_cols = ["firstname", "lastname", "busname", "dob"]

df_filtered.loc[:, "new_id"]  = df_filtered[id_cols].astype(str).agg("-".join, axis=1)

In [26]:
###################################
# filter dataset on most recent exclusion only to get 1 per row
###################################

# Sort so most recent date is first
df_filtered = df_filtered.sort_values(["new_id", "excldate"], ascending=[True, False])

# Keep only the most recent row per ID
df_filtered = df_filtered.groupby("new_id", as_index=False).head(1)

In [27]:
##############################################################
# Return a dataset that has null NPIs to possibly add to later
##############################################################
df_with_null_npi = df_filtered

df_with_null_npi.to_csv("/dsa/groups/casestudycf25/team02/bronze/leie_with_null_npi_clean.csv", index=False)
df_with_null_npi.shape

(7692, 20)

In [28]:
# Check the newly created CSV
temp = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/leie_with_null_npi_clean.csv")

temp.shape

(7692, 20)

In [29]:
###################################
# filter dataset where there is only valid NPIs., exclude nulls
###################################
df_valid_npi = df_filtered[df_filtered["npi"].notna()].copy()

In [30]:
print("Check if there are any duplicate NPIs in final dataset: ")
df_valid_npi["npi"].duplicated().any()

Check if there are any duplicate NPIs in final dataset: 


False

In [31]:
print(f"New Dataset Shape After Filtering for Valid NPIs: {df_valid_npi.shape}")

New Dataset Shape After Filtering for Valid NPIs: (1884, 20)


In [32]:
# Add in fraud flag where every row is 1
df_valid_npi.loc[:, "fraud_flag"] = 1

In [33]:
type_dummies = pd.get_dummies(df_valid_npi["excltype"], prefix="excl")

# Attach back to df
df_valid_npi = df_valid_npi.join(type_dummies)

In [34]:
df_valid_npi.drop(columns = ["lastname" , "firstname" , "midname" , "busname", 
                            "dob","upin",
                             "address", "city", "state", "zip"], inplace = True)

In [35]:
df_valid_npi.head(1)

,general,specialty,npi,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime,new_id,fraud_flag,excl_1128a1,excl_1128a2,excl_1128a3,excl_1128b5,excl_1128b6,excl_1128b7,excl_brch cia
77491,IND- LIC HC SERV PRO,DENTIST,1.861529e+09,1128a3,2024-03-20,1.0,1.0,[1128a3],1.0,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0


In [36]:
col_totals = df_valid_npi.select_dtypes(include="number").sum()
print("All Column Totals")
print("-" *50)
print(col_totals.apply(lambda x: f"{x:.0f}"))

All Column Totals
--------------------------------------------------
npi                            2820577712190
num_exclusions_alltime                  1942
num_exclusion_types_alltime             1924
num_addresses_alltime                   1936
fraud_flag                              1884
excl_1128a1                             1343
excl_1128a2                              142
excl_1128a3                              319
excl_1128b5                               10
excl_1128b6                                3
excl_1128b7                               66
excl_brch cia                              1
dtype: object


In [37]:
##############################################################
# Return a dataset that is clean for modeling use
##############################################################

df_valid_npi.to_csv("/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv", index=False)
df_valid_npi.shape

(1884, 18)

In [38]:
# Check the newly created CSV
temp = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv")

temp.shape

(1884, 18)